# chunking

In [3]:
import json
with open(r"..\data\processed/chunks.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

split_records = [r for r in records if r["was_split"]]
print("Total chunks:", len(records))
print("Chunks from split records:", len(split_records))
print("Max chunk_index seen:", max(r["chunk_index"] for r in records))

Total chunks: 7064
Chunks from split records: 3776
Max chunk_index seen: 7


In [5]:
from pathlib import Path
print(Path.cwd().parent)

d:\Programing\Github\Persian-Medical-RAG-Chatbot


# retrievel

In [6]:
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


PROJECT_ROOT = Path.cwd().parent
FAISS_DIR = PROJECT_ROOT / "data" / "processed" / "faiss_index"

# Must match the exact model used in build_index.py - mixing embedding
# models between indexing and querying produces meaningless results.
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# Number of chunks to retrieve per query (k=5 per the project proposal)
TOP_K = 5

def load_vectorstore() -> FAISS:
    if not FAISS_DIR.exists():
        raise FileNotFoundError(
            f"FAISS index not found at: {FAISS_DIR}\n"
            f"Run build_index.py first to generate it."
        )

    embedding = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )

    return FAISS.load_local(
        str(FAISS_DIR),
        embedding,
        allow_dangerous_deserialization=True,
    )


def search(vectorstore: FAISS, query: str, k: int = TOP_K):
    """Return the top-k most relevant chunks for a given query.

    Note: multilingual-e5 models require a "query: " prefix on search
    queries (documents were indexed with a "passage: " prefix in
    build_index.py). This is a requirement of the model itself.
    """
    prefixed_query = "query: " + query
    results = vectorstore.similarity_search_with_score(prefixed_query, k=k)
    return results


def print_results(query: str, results) -> None:
    print(f"\nQuery: {query}")
    print(f"Top {len(results)} results:\n")
    for rank, (doc, score) in enumerate(results, start=1):
        print(f"[{rank}] score={score:.4f} | drug={doc.metadata.get('drug_name')}")
        print(f"    {doc.page_content}")
        print()

vectorstore = load_vectorstore()

C:\Users\Saraye Tel\AppData\Local\Temp\ipykernel_4732\528627244.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [12]:
sample_queries = [
    "آسپرین برای سردرد خوبه؟",
    "مصرف قرص فلوکستین چه عوارضی داره؟",
    "آیا شربت لاکتولوز برای یبوست کودکان مناسب است؟",
]
for q in sample_queries:
    results = search(vectorstore, q)
    print_results(q, results)


Query: آسپرین برای سردرد خوبه؟
Top 5 results:

[1] score=0.2884 | drug=Acetaminophen-Ibuprofen-Caffeine
    passage: دارو: Acetaminophen-Ibuprofen-Caffeine | سؤال: سلام. چندروزی است سردرد شدیدی می‌گیرم طوری که فشار زیادی روی چشم چپم میاره. چند روز بیمارستان بستری بودم تمام آزمایشها ام. آر. ای. سیتیزن اسکن گرفتن هیچی متوجه نشدند. الان دکتر دیگه رفتم فارژزیک تجویز کرده آیا اثری داره …. لطفا راهنماییم کنید | پاسخ: این دارو مسکن است و خاصیت ضد درد نسبتا قوی دارد ولی اگر بیماری زمینه‌ای مانند میگرن داشته باشید درمان‌کننده علت نخواهد بود. اگر نتایج آزمایشات و سی تی اسکن مشکل خاصی نشان نداده به نظر نمی‌رسد مشکل خاصی وجود داشته باشد.

[2] score=0.2895 | drug=Sumatriptan-Naproxen
    passage: دارو: Sumatriptan-Naproxen | سؤال: سلام وقت شما بخیر چند سالی هست سر درد دارم و اوایل از داروهای مقل نوافن استفاده می‌کردم بعد متوجه شدم میگرن هست و با مراجعه به دکتر مغز و اعصاب از دپاکین استفاده می‌کردم که باعث می‌شد سردردم شدید‌تر بشه به دکتر مراجعه کردم داروها رو قطع کردم و از دارو ریزتریپتان استفاده 

In [8]:
sample_queries = [
    "دوز مصرف سفیکسیم برای کودکان چقدره؟",
    "آیا مصرف ایبوپروفن در دوران بارداری خطرناکه؟",
    "قرص فاموتیدین با چه داروهایی تداخل داره؟",
    "سیپروفلوکساسین برای عفونت ادراری چند روز باید مصرف بشه؟",
    "تفاوت فلوکستین و فلووکسامین چیه؟",  # تست تشخیص دو دارو شبیه‌هم
    "آیا موپیروسین برای زخم صورت هم استفاده میشه؟",
    "کلردیازپوکساید چه عوارض جانبی‌ای داره؟",
    "آیا میشه تئوفیلین رو با قهوه مصرف کرد؟",
]

for q in sample_queries:
    results = search(vectorstore, q)
    print_results(q, results)


Query: دوز مصرف سفیکسیم برای کودکان چقدره؟
Top 5 results:

[1] score=0.2567 | drug=Cefixime
    passage: دارو: Cefixime | سؤال: سلام، وقت بخیر. کودک ۶ ساله ۳۰ کیلویی به تجویز متخصص کودکان برای درمان سرفه خلطی و تب، یک شیشه شربت سفیکسیم را به صورت هر ۱۲ ساعت ۶ سی‌سی استفاده کرده است. اما هنوز ۷ روز نشده دارو تمام‌شده. آیا نیازی به تهیه شربت دیگر و ادامه درمان تا ۷ یا ۱۰ روز هست؟ چند روز؟ قبلا از راهنمایی شما متشکرم. | پاسخ: اگر نیاز به مقدار بیشتر آنتی بیوتیک وجود داشت پزشک تجویز کرده بود. نیازی به تهیه سفکیسم اضافه نیست.

[2] score=0.2653 | drug=Cefixime
    passage: دارو: Cefixime | سؤال: با سلام.. دکتر برای من سفکسیم ۴۰۰ تجویز کردن اما دستور مصرفش نوشته روزی دوبار.. من فکر می‌کنم این دوز خیلی بالاس بخاطر همین هنوز مصرف نکردم.. قبلا از ایتروکونازول ۱۰۰ استفاده کردم و دچار تنگی نفس شدم …لطفا راهنماییم کنید. مچکرم | پاسخ: دوز داروها توسط پزشک تجویز می‌شود که بستگی به شدت و نوع بیماری، وزن بیمار و سایر عوامل زمینه‌ای است. معمولا دوز سفکیسم برای بزرگسالان روزانه ۴۰۰ میلی‌گرم است ولی به ن

# generation

In [9]:
from pathlib import Path
import ollama

GENERATION_MODEL = "qwen2.5:3b-instruct"

SYSTEM_PROMPT = """تو یک دستیار اطلاعات دارویی هستی.

فقط بر اساس اطلاعات موجود در بخش «زمینه» (Context) پاسخ بده.
اگر پاسخ سؤال در زمینه وجود ندارد، اطلاعات را حدس نزن و صریحاً اعلام کن
که اطلاعات کافی در منابع موجود نیست.

پاسخ را به زبان فارسی، واضح و مختصر ارائه کن.

در مورد تشخیص قطعی بیماری یا تغییر خودسرانه‌ی دوز دارو، توصیه‌ی قطعی
ارائه نکن و کاربر را به مراجعه به پزشک یا داروساز ارجاع بده."""


def build_context(results) -> str:
    """Format retrieved (doc, score) pairs into a single context block."""
    parts = []
    for i, (doc, _score) in enumerate(results, start=1):
        drug = doc.metadata.get("drug_name", "نامشخص")
        # Strip the "passage: " prefix added during indexing - it's an
        # embedding-model requirement, not something the LLM should see.
        text = doc.page_content.removeprefix("passage: ")
        parts.append(f"[منبع {i} - دارو: {drug}]\n{text}")
    return "\n\n".join(parts)


def generate_answer(question: str, results) -> str:
    """Generate a Persian answer from the user's question and retrieved chunks.

    Requires the Ollama app to be running in the background (it starts
    automatically after installation on most systems).
    """
    context = build_context(results)
    prompt = f"زمینه (Context):\n{context}\n\nسؤال کاربر:\n{question}"

    response = ollama.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0.2},  # low temperature: grounded, consistent answers
    )

    return response["message"]["content"]


In [10]:
question = "ازیترومایسین کاربردش چیه؟"
vectorstore = load_vectorstore()
results = search(vectorstore, question)
answer = generate_answer(question, results)

print(answer)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

ازیترومایسین برای علاوهای مختلفی مانند بیماری‌های ریهی، سینوسی، و سیستم خونی استفاده می‌شود. در برخی موارد، می‌تواند برای کاهش ضربان قلبی نیز استفاده می‌شود.


# rag_test

In [11]:

from pathlib import Path
import sys

# Allow importing retrieve.py and generator.py from the retrieval/
# and generation/ folders without turning the whole project into a
# formal installable package - fine for a small team project.
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src" / "retrieval"))
sys.path.append(str(PROJECT_ROOT / "src" / "generation"))


def answer_question(vectorstore, question: str) -> None:
    results = search(vectorstore, question)
    answer = generate_answer(question, results)

    print(f"\nسؤال: {question}")
    print(f"پاسخ: {answer}\n")
    print("منابع استفاده‌شده:")
    for i, (doc, score) in enumerate(results, start=1):
        print(f"  [{i}] {doc.metadata.get('drug_name')} (score={score:.4f})")
    print("-" * 60)


if __name__ == "__main__":
    vectorstore = load_vectorstore()

    test_questions = [
        "آسپرین برای سردرد خوبه؟",
        "تفاوت فلوکستین و فلووکسامین چیه؟",
        "دوز مصرف سفیکسیم برای کودکان چقدره؟",
    ]

    for q in test_questions:
        answer_question(vectorstore, q)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


سؤال: آسپرین برای سردرد خوبه؟
پاسخ: آسپرین برای سردرد خوبه می‌تواند باشد ولی باید با پزشک مراجعه کنید تا بتوانید بهترین درمان برای شما پیشنهاد شود.

منابع استفاده‌شده:
  [1] Acetaminophen-Ibuprofen-Caffeine (score=0.2884)
  [2] Sumatriptan-Naproxen (score=0.2895)
  [3] Sodium-Valproate (score=0.2898)
  [4] Enoxaparin-Sodium (score=0.2904)
  [5] Aspirin (score=0.2909)
------------------------------------------------------------

سؤال: تفاوت فلوکستین و فلووکسامین چیه؟
پاسخ: فلوکستین و فلووکسامین داروهای ضد افسردگی هستند و از نظر عملکرد هر دو مهارکننده انتخابی سروتونین هستند. ولی فلووکسامین معمولاً کاهش میل جنسی ایجاد می‌کند.

منابع استفاده‌شده:
  [1] Fluvoxamine (score=0.2285)
  [2] Fluvoxamine (score=0.3055)
  [3] Bupropion (score=0.3056)
  [4] Fluoxetine (score=0.3102)
  [5] Fluvoxamine (score=0.3145)
------------------------------------------------------------

سؤال: دوز مصرف سفیکسیم برای کودکان چقدره؟
پاسخ: دوز مصرف سفکسیم برای کودکان توسط پزشک تجویز می‌شود. معمولاً سفکسیم برای بزرگ